# Quick Experiment

Run a minimal ingestion and retrieval flow against sample documents.

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [2]:
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_retrieval_pipeline
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEERS

config = RetrievalConfig.from_env().with_overrides(chunk_size=2000, chunk_overlap=20, top_k=3)
ingest_service, search_service = create_retrieval_pipeline(config)

In [4]:
documents = ingest_service.add_texts(
    texts=[item["text"] for item in SAMPLE_ENGINEERS],
    metadatas=[item["metadata"] for item in SAMPLE_ENGINEERS],
)
len(documents), documents[0]

(9,
 Document(metadata={'engineer_id': 'eng-001', 'grade': 'SENIOR', 'status': 'AVAILABLE', 'engineer_role': 'DEVELOPER', 'engineer_type': 'FULL_TIME'}, page_content='[기술스택]\n        Java / Spring Boot / PostgreSQL / Redis / Docker\n        \n        [경력]\n        8년차 백엔드 개발자.\n\n        [프로젝트 경험]\n        현대모비스(2023~2025): ERP 재고관리 모듈 개발. Spring Batch로 야간 정산 처리 자동화. 포지션: 백엔드 리드.\n        LG CNS(2021~2023): 제조업 MES 시스템 API 개발. Oracle DB 쿼리 최적화로 조회속도 40% 개선. 포지션: 백엔드 개발자.\n        삼성SDS(2017~2021): 물류 ERP 시스템 개발. Spring Boot 기반 REST API 설계. 포지션: 백엔드 개발자.'))

In [5]:
query = "PL"
results = search_service.search(query, top_k=2)
[(round(item.score, 4), item.document.metadata, item.document.page_content) for item in results]

[(0.8083,
  {'engineer_id': 'pln-001',
   'grade': 'SENIOR',
   'status': 'AVAILABLE',
   'engineer_role': 'PLANNER',
   'engineer_type': 'FULL_TIME'},
  '[기술스택]\n        Confluence / Slack / Notion / GA4 / SQL\n\n        [경력]\n        9년차 서비스 기획자(PM).\n\n        [프로젝트 경험]\n        당근마켓(2023~2025): 지역 커뮤니티 신규 피드 서비스 기획 및 런칭. MAU 20% 증대.\n        직방(2020~2023): 부동산 매물 관리 시스템 백오피스 기획 및 프로세스 자동화.\n        이커머스 기업(2016~2020): 주문/결제 서비스 운영 기획. 포지션: PM.'),
 (0.8083,
  {'engineer_id': 'pln-001',
   'grade': 'SENIOR',
   'status': 'AVAILABLE',
   'engineer_role': 'PLANNER',
   'engineer_type': 'FULL_TIME'},
  '[기술스택]\n        Confluence / Slack / Notion / GA4 / SQL\n\n        [경력]\n        9년차 서비스 기획자(PM).\n\n        [프로젝트 경험]\n        당근마켓(2023~2025): 지역 커뮤니티 신규 피드 서비스 기획 및 런칭. MAU 20% 증대.\n        직방(2020~2023): 부동산 매물 관리 시스템 백오피스 기획 및 프로세스 자동화.\n        이커머스 기업(2016~2020): 주문/결제 서비스 운영 기획. 포지션: PM.')]

In [3]:
# 벡터 DB 전체 초기화
ingest_service._vector_store._store.delete(delete_all=True)
print("done")

done
